# TP Tutoriel : GPT, LLMs et Capacités Émergentes

**ITS3 - Natural Language Processing**  
**Auteur :** Kenza BOUZIDI  
**Contact :** kenza.bouzidi.ext@u-pec.fr

---

## Objectifs du TP

Dans ce TP tutoriel, vous allez :
1. Comprendre le fonctionnement de la **tokenization BPE**
2. Implémenter le mécanisme d'**attention causale** (masked self-attention)
3. Explorer les **capacités des LLMs** via l'API OpenAI/HuggingFace
4. Expérimenter avec différentes techniques de **prompting**
5. Implémenter un système **RAG** simple

---

## Configuration de l'environnement

Commençons par installer et importer les bibliothèques nécessaires.

In [ ]:
# Installation des dépendances (décommenter si nécessaire)
# !pip install transformers torch tiktoken openai sentence-transformers faiss-cpu numpy matplotlib

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from collections import defaultdict, Counter
import re

# Pour la tokenization
import tiktoken
from transformers import GPT2Tokenizer, GPT2LMHeadModel

print(" Bibliothèques importées avec succès !")

---

# Partie 1 : Tokenization BPE (Byte Pair Encoding)

## 1.1 Comprendre BPE

Le **Byte Pair Encoding** est l'algorithme de tokenization utilisé par GPT. Il permet de :
- Avoir un vocabulaire de taille fixe
- Gérer les mots inconnus (OOV)
- Partager des sous-unités entre mots similaires

### Algorithme :
1. Commencer avec des caractères individuels
2. Compter les paires adjacentes
3. Fusionner la paire la plus fréquente
4. Répéter jusqu'à la taille de vocabulaire désirée

In [ ]:
class SimpleBPE:
    """
    Implémentation simplifiée de l'algorithme BPE pour comprendre son fonctionnement.
    """
    
    def __init__(self):
        self.vocab = {}
        self.merges = []
    
    def get_stats(self, corpus):
        """
        Compte les fréquences de toutes les paires de tokens adjacents.
        
        Args:
            corpus: dict {mot_tokenisé: fréquence}
        Returns:
            Counter des paires
        """
        pairs = Counter()
        for word, freq in corpus.items():
            symbols = word.split()
            for i in range(len(symbols) - 1):
                pairs[(symbols[i], symbols[i+1])] += freq
        return pairs
    
    def merge_vocab(self, pair, corpus):
        """
        Fusionne une paire de tokens dans tout le corpus.
        
        Args:
            pair: tuple (token1, token2) à fusionner
            corpus: dict du corpus actuel
        Returns:
            Nouveau corpus avec la paire fusionnée
        """
        new_corpus = {}
        bigram = ' '.join(pair)
        replacement = ''.join(pair)
        
        for word, freq in corpus.items():
            new_word = word.replace(bigram, replacement)
            new_corpus[new_word] = freq
        
        return new_corpus
    
    def fit(self, words_freq, num_merges=10, verbose=True):
        """
        Entraîne le tokenizer BPE.
        
        Args:
            words_freq: dict {mot: fréquence}
            num_merges: nombre d'itérations de fusion
            verbose: afficher les étapes
        """
        # Initialisation : séparer chaque mot en caractères + symbole de fin
        corpus = {}
        for word, freq in words_freq.items():
            # Ajouter des espaces entre caractères et le symbole de fin '_'
            tokenized = ' '.join(list(word)) + ' _'
            corpus[tokenized] = freq
        
        if verbose:
            print("=" * 60)
            print("CORPUS INITIAL :")
            for word, freq in corpus.items():
                print(f"  '{word}' : {freq}")
            print("=" * 60)
        
        # Itérations de fusion
        for i in range(num_merges):
            pairs = self.get_stats(corpus)
            
            if not pairs:
                break
            
            # Trouver la paire la plus fréquente
            best_pair = max(pairs, key=pairs.get)
            best_freq = pairs[best_pair]
            
            # Fusionner
            corpus = self.merge_vocab(best_pair, corpus)
            self.merges.append(best_pair)
            
            if verbose:
                print(f"\n Itération {i+1}:")
                print(f"   Paire la plus fréquente : {best_pair} (fréquence: {best_freq})")
                print(f"   Fusion : '{best_pair[0]}' + '{best_pair[1]}' → '{''.join(best_pair)}'")
                print(f"   Corpus mis à jour :")
                for word, freq in corpus.items():
                    print(f"      '{word}' : {freq}")
        
        # Construire le vocabulaire final
        self.vocab = set()
        for word in corpus.keys():
            for token in word.split():
                self.vocab.add(token)
        
        if verbose:
            print("\n" + "=" * 60)
            print(f"VOCABULAIRE FINAL ({len(self.vocab)} tokens) :")
            print(sorted(self.vocab))
            print("=" * 60)
        
        return corpus
    
    def tokenize(self, word):
        """
        Tokenise un mot en utilisant les règles de fusion apprises.
        
        Args:
            word: mot à tokeniser
        Returns:
            Liste de tokens
        """
        # Commencer par les caractères
        tokens = list(word) + ['_']
        
        # Appliquer les fusions dans l'ordre
        for pair in self.merges:
            i = 0
            while i < len(tokens) - 1:
                if tokens[i] == pair[0] and tokens[i+1] == pair[1]:
                    tokens = tokens[:i] + [''.join(pair)] + tokens[i+2:]
                else:
                    i += 1
        
        return tokens

### Exemple pratique : Entraînement BPE

In [ ]:
# Corpus d'exemple
corpus_exemple = {
    'low': 5,
    'lower': 2,
    'newest': 6,
    'widest': 3
}

print("Corpus d'entraînement :")
for word, freq in corpus_exemple.items():
    print(f"   '{word}' apparaît {freq} fois")

# Entraîner le BPE
bpe = SimpleBPE()
final_corpus = bpe.fit(corpus_exemple, num_merges=5, verbose=True)

### Test de tokenization

In [ ]:
# Tester la tokenization sur de nouveaux mots
test_words = ['lowest', 'newer', 'widening', 'test']

print("\n Test de tokenization :")
print("-" * 40)
for word in test_words:
    tokens = bpe.tokenize(word)
    print(f"'{word}' → {tokens}")

## 1.2 Tokenizer GPT-2 (tiktoken)

Voyons maintenant comment fonctionne le vrai tokenizer de GPT-2/GPT-3.

In [ ]:
# Charger le tokenizer GPT-2
enc = tiktoken.get_encoding("gpt2")

# Exemples de tokenization
examples = [
    "Hello, world!",
    "Bonjour le monde!",
    "The quick brown fox jumps over the lazy dog.",
    "GPT-4 is a large language model.",
    "Tokenization is fascinating!",
    "supercalifragilisticexpialidocious"  # Mot très long
]

print(" Tokenization avec GPT-2 (tiktoken)")
print("=" * 70)

for text in examples:
    tokens = enc.encode(text)
    decoded_tokens = [enc.decode([t]) for t in tokens]
    
    print(f"\nTexte: '{text}'")
    print(f"  Nombre de tokens: {len(tokens)}")
    print(f"  Token IDs: {tokens}")
    print(f"  Tokens décodés: {decoded_tokens}")

In [ ]:
# Visualisation : ratio tokens/mots pour différentes langues
texts = {
    'Anglais': "The transformer architecture has revolutionized natural language processing.",
    'Français': "L'architecture transformer a révolutionné le traitement du langage naturel.",
    'Allemand': "Die Transformer-Architektur hat die Verarbeitung natürlicher Sprache revolutioniert.",
    'Code Python': "def hello_world():\n    print('Hello, World!')\n    return True"
}

print(" Comparaison du nombre de tokens par langue")
print("=" * 60)

for lang, text in texts.items():
    num_tokens = len(enc.encode(text))
    num_words = len(text.split())
    ratio = num_tokens / num_words
    print(f"{lang}:")
    print(f"  Mots: {num_words}, Tokens: {num_tokens}, Ratio: {ratio:.2f}")
    print()

---

# Partie 2 : Mécanisme d'Attention Causale

## 2.1 Implémentation de l'attention

L'attention est le cœur de l'architecture Transformer. Dans GPT, on utilise l'**attention causale** (masked self-attention) où chaque token ne peut voir que les tokens précédents.

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Calcule l'attention scaled dot-product.
    
    Formule: Attention(Q, K, V) = softmax(QK^T / sqrt(d_k)) * V
    
    Args:
        Q: Queries [batch_size, seq_len, d_k]
        K: Keys [batch_size, seq_len, d_k]
        V: Values [batch_size, seq_len, d_v]
        mask: Masque optionnel [seq_len, seq_len]
    
    Returns:
        output: Résultat de l'attention [batch_size, seq_len, d_v]
        attention_weights: Poids d'attention [batch_size, seq_len, seq_len]
    """
    d_k = Q.size(-1)
    
    # Étape 1: Calcul des scores QK^T
    scores = torch.matmul(Q, K.transpose(-2, -1))
    print(f"Scores QK^T (avant scaling):\n{scores[0]}\n")
    
    # Étape 2: Scaling par sqrt(d_k)
    scores = scores / np.sqrt(d_k)
    print(f"Scores après scaling (÷√{d_k}):\n{scores[0]}\n")
    
    # Étape 3: Application du masque (si fourni)
    if mask is not None:
        scores = scores.masked_fill(mask == 0, float('-inf'))
        print(f"Scores après masquage:\n{scores[0]}\n")
    
    # Étape 4: Softmax
    attention_weights = F.softmax(scores, dim=-1)
    print(f"Poids d'attention (après softmax):\n{attention_weights[0]}\n")
    
    # Étape 5: Multiplication par V
    output = torch.matmul(attention_weights, V)
    
    return output, attention_weights

In [ ]:
def create_causal_mask(seq_len):
    """
    Crée un masque causal (triangulaire inférieur).
    
    Le masque empêche chaque position de voir les positions futures.
    Position i peut seulement voir positions 0, 1, ..., i
    
    Args:
        seq_len: Longueur de la séquence
    
    Returns:
        Masque binaire [seq_len, seq_len]
    """
    mask = torch.tril(torch.ones(seq_len, seq_len))
    return mask

# Visualiser le masque causal
seq_len = 5
mask = create_causal_mask(seq_len)

print("Masque causal (1 = peut voir, 0 = masqué) :")
print(mask)

# Visualisation graphique
plt.figure(figsize=(6, 5))
plt.imshow(mask, cmap='Blues')
plt.colorbar(label='Peut voir')
plt.xlabel('Position clé (K)')
plt.ylabel('Position requête (Q)')
plt.title('Masque Causal pour GPT')
for i in range(seq_len):
    for j in range(seq_len):
        color = 'white' if mask[i, j] == 1 else 'black'
        plt.text(j, i, '✓' if mask[i, j] == 1 else '✗', 
                 ha='center', va='center', color=color, fontsize=14)
plt.tight_layout()
plt.show()

### Exemple complet d'attention causale

In [ ]:
# Exemple avec une séquence de 4 tokens
print("=" * 70)
print("EXEMPLE : Attention Causale sur 4 tokens")
print("=" * 70)

# Simuler des embeddings (batch_size=1, seq_len=4, d_model=3)
torch.manual_seed(42)
seq_len = 4
d_model = 3

# Q, K, V sont les mêmes pour la self-attention
X = torch.randn(1, seq_len, d_model)
Q = K = V = X

print(f"\n Input X (représentant 4 tokens) :")
print(f"   Forme: {X.shape}")
print(f"   Valeurs:\n{X[0]}\n")

# Créer le masque causal
mask = create_causal_mask(seq_len)

# Calculer l'attention
print("\n CALCUL DE L'ATTENTION ÉTAPE PAR ÉTAPE :\n")
output, attention_weights = scaled_dot_product_attention(Q, K, V, mask)

print(f" Output (contextualisé) :\n{output[0]}")

In [ ]:
# Visualisation des poids d'attention
plt.figure(figsize=(8, 6))
plt.imshow(attention_weights[0].detach().numpy(), cmap='viridis')
plt.colorbar(label='Poids d\'attention')
plt.xlabel('Position clé (source)')
plt.ylabel('Position requête (cible)')
plt.title('Poids d\'Attention Causale')

# Annoter avec les valeurs
for i in range(seq_len):
    for j in range(seq_len):
        val = attention_weights[0, i, j].item()
        color = 'white' if val > 0.3 else 'black'
        plt.text(j, i, f'{val:.2f}', ha='center', va='center', color=color)

plt.xticks(range(seq_len), [f'Token {i}' for i in range(seq_len)])
plt.yticks(range(seq_len), [f'Token {i}' for i in range(seq_len)])
plt.tight_layout()
plt.show()

print("\n Interprétation :")
print("   - Token 0 ne regarde que lui-même (100%)")
print("   - Token 1 regarde Token 0 et lui-même")
print("   - Token 3 peut regarder tous les tokens précédents")
print("   - Les positions futures sont masquées (poids = 0)")

## 2.2 Multi-Head Attention

In [ ]:
class MultiHeadAttention(nn.Module):
    """
    Multi-Head Attention comme dans le Transformer.
    
    Plusieurs têtes d'attention permettent au modèle d'apprendre
    différents types de relations entre les tokens.
    """
    
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, "d_model doit être divisible par num_heads"
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        
        # Projections linéaires pour Q, K, V
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        
        # Projection de sortie
        self.W_o = nn.Linear(d_model, d_model)
    
    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.size()
        
        # 1. Projections linéaires
        Q = self.W_q(x)
        K = self.W_k(x)
        V = self.W_v(x)
        
        # 2. Reshape pour multi-head: [batch, seq, d_model] -> [batch, heads, seq, d_k]
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # 3. Scaled dot-product attention
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)
        
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        attention_weights = F.softmax(scores, dim=-1)
        context = torch.matmul(attention_weights, V)
        
        # 4. Concaténer les têtes: [batch, heads, seq, d_k] -> [batch, seq, d_model]
        context = context.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        
        # 5. Projection finale
        output = self.W_o(context)
        
        return output, attention_weights

# Test
d_model = 64
num_heads = 4
seq_len = 8
batch_size = 2

mha = MultiHeadAttention(d_model, num_heads)
x = torch.randn(batch_size, seq_len, d_model)
mask = create_causal_mask(seq_len).unsqueeze(0).unsqueeze(0)  # [1, 1, seq, seq]

output, attn_weights = mha(x, mask)

print(f"   Multi-Head Attention")
print(f"   Input shape: {x.shape}")
print(f"   Output shape: {output.shape}")
print(f"   Attention weights shape: {attn_weights.shape}")
print(f"   (batch={batch_size}, heads={num_heads}, seq={seq_len}, seq={seq_len})")

---

# Partie 3 : Utilisation des LLMs

## 3.1 Génération de texte avec GPT-2

In [ ]:
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Charger GPT-2
print("⏳ Chargement de GPT-2...")
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2')
model.eval()
print(" GPT-2 chargé !")

# Informations sur le modèle
num_params = sum(p.numel() for p in model.parameters())
print(f"\n Statistiques du modèle :")
print(f"   Nombre de paramètres : {num_params:,} ({num_params/1e6:.1f}M)")
print(f"   Vocabulaire : {tokenizer.vocab_size:,} tokens")

In [ ]:
def generate_text(prompt, max_length=50, temperature=1.0, top_k=50, num_return=1):
    """
    Génère du texte avec GPT-2.
    
    Args:
        prompt: Texte de départ
        max_length: Longueur maximale
        temperature: Contrôle la créativité (0 = déterministe, >1 = plus aléatoire)
        top_k: Nombre de tokens les plus probables à considérer
        num_return: Nombre de séquences à générer
    """
    inputs = tokenizer.encode(prompt, return_tensors='pt')
    
    # Générer
    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_length=max_length,
            temperature=temperature,
            top_k=top_k,
            num_return_sequences=num_return,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    # Décoder
    generated_texts = []
    for output in outputs:
        text = tokenizer.decode(output, skip_special_tokens=True)
        generated_texts.append(text)
    
    return generated_texts

# Test de génération
prompt = "Artificial intelligence is"
print(f" Prompt : '{prompt}'")
print("\n" + "=" * 60)

# Différentes températures
for temp in [0.3, 0.7, 1.2]:
    print(f"\n Température = {temp}")
    texts = generate_text(prompt, max_length=60, temperature=temp)
    print(f"   {texts[0]}")

## 3.2 Visualisation des probabilités de tokens

In [ ]:
def get_next_token_probs(text, top_k=10):
    """
    Affiche les probabilités des prochains tokens possibles.
    """
    inputs = tokenizer.encode(text, return_tensors='pt')
    
    with torch.no_grad():
        outputs = model(inputs)
        logits = outputs.logits[0, -1, :]  # Dernier token
        probs = F.softmax(logits, dim=-1)
    
    # Top-k tokens
    top_probs, top_indices = torch.topk(probs, top_k)
    
    print(f"\n Top {top_k} prochains tokens pour : '{text}'")
    print("-" * 50)
    
    tokens = []
    probabilities = []
    
    for i, (prob, idx) in enumerate(zip(top_probs, top_indices)):
        token = tokenizer.decode([idx])
        tokens.append(repr(token))
        probabilities.append(prob.item())
        print(f"   {i+1}. {repr(token):15} : {prob.item()*100:5.2f}%")
    
    # Visualisation
    plt.figure(figsize=(10, 5))
    bars = plt.barh(range(len(tokens)), probabilities[::-1])
    plt.yticks(range(len(tokens)), tokens[::-1])
    plt.xlabel('Probabilité')
    plt.title(f'Probabilités des prochains tokens après "{text}"')
    plt.tight_layout()
    plt.show()

# Exemples
get_next_token_probs("The capital of France is")
get_next_token_probs("Machine learning is a")

---

# Partie 4 : Techniques de Prompting

## 4.1 Zero-shot vs Few-shot

Utilisons un modèle plus puissant via l'API HuggingFace (ou simulons les résultats).

In [ ]:
# Simulation de réponses LLM pour démonstration
# (En pratique, vous utiliseriez l'API OpenAI ou HuggingFace)

def simulate_llm_response(prompt, mode="explain"):
    """
    Simule une réponse de LLM pour les exemples de prompting.
    En pratique, remplacez par un appel API réel.
    """
    print(" PROMPT :")
    print("-" * 50)
    print(prompt)
    print("-" * 50)
    print("\n RÉPONSE ATTENDUE DU LLM :")
    print("(Simulation - utilisez l'API pour de vrais résultats)")
    print()

# Zero-shot
print("=" * 70)
print("ZERO-SHOT PROMPTING")
print("=" * 70)

zero_shot_prompt = """Classifie le sentiment du texte suivant.
Réponds uniquement par : Positif, Négatif, ou Neutre.

Texte : "Ce restaurant propose une cuisine délicieuse mais le service est vraiment trop lent."

Sentiment :"""

simulate_llm_response(zero_shot_prompt)
print("→ Mitigé/Neutre (aspects positifs et négatifs)")

In [ ]:
# Few-shot
print("\n" + "=" * 70)
print("FEW-SHOT PROMPTING")
print("=" * 70)

few_shot_prompt = """Classifie le sentiment des avis de restaurants.

Avis : "Excellente soirée ! La nourriture était parfaite et le personnel adorable."
Sentiment : Positif

Avis : "Très déçu. Plat froid, serveur désagréable, je ne reviendrai pas."
Sentiment : Négatif

Avis : "Repas correct, rien d'exceptionnel. Prix dans la moyenne."
Sentiment : Neutre

Avis : "Ce restaurant propose une cuisine délicieuse mais le service est vraiment trop lent."
Sentiment :"""

simulate_llm_response(few_shot_prompt)
print("→ Mitigé (ou Neutre selon le modèle)")

## 4.2 Chain-of-Thought (CoT)

In [ ]:
print("=" * 70)
print("CHAIN-OF-THOUGHT PROMPTING")
print("=" * 70)

# Sans CoT
print("\n SANS Chain-of-Thought :")
print("-" * 50)

no_cot_prompt = """Question : Un magasin vend des pommes à 2€ pièce. 
Marie achète 4 pommes et donne un billet de 20€. 
Elle utilise ensuite la moitié de sa monnaie pour acheter un magazine.
Combien lui reste-t-il ?

Réponse :"""

print(no_cot_prompt)
print("\n→ Un petit modèle pourrait répondre incorrectement : '10€' ou '12€'")

# Avec CoT
print("\n\n AVEC Chain-of-Thought :")
print("-" * 50)

cot_prompt = """Question : Un magasin vend des pommes à 2€ pièce. 
Marie achète 4 pommes et donne un billet de 20€. 
Elle utilise ensuite la moitié de sa monnaie pour acheter un magazine.
Combien lui reste-t-il ?

Résolvons étape par étape :

Étape 1 : Calculer le coût des pommes
- Prix d'une pomme : 2€
- Nombre de pommes : 4
- Coût total : 4 × 2€ = 8€

Étape 2 : Calculer la monnaie rendue
- Marie paie : 20€
- Coût : 8€
- Monnaie : 20€ - 8€ = 12€

Étape 3 : Calculer le reste après le magazine
- Marie utilise la moitié de 12€
- Magazine : 12€ ÷ 2 = 6€
- Reste : 12€ - 6€ = 6€

Réponse finale : Marie a 6€."""

print(cot_prompt)

In [ ]:
# Zero-shot CoT (la technique magique !)
print("\n" + "=" * 70)
print("ZERO-SHOT CHAIN-OF-THOUGHT")
print("=" * 70)

zero_shot_cot = """Question : Un magasin vend des pommes à 2€ pièce. 
Marie achète 4 pommes et donne un billet de 20€. 
Elle utilise ensuite la moitié de sa monnaie pour acheter un magazine.
Combien lui reste-t-il ?

Réfléchissons étape par étape."""

print(zero_shot_cot)
print("\n→ En ajoutant simplement 'Réfléchissons étape par étape', ")
print("  le modèle génère automatiquement le raisonnement !")

---

# Partie 5 : RAG (Retrieval-Augmented Generation)

## 5.1 Implémentation d'un RAG simple

RAG combine la recherche d'information avec la génération pour réduire les hallucinations.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Charger un modèle d'embeddings
print(" Chargement du modèle d'embeddings...")
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
print(" Modèle chargé !")

In [ ]:
class SimpleRAG:
    """
    Implémentation simple d'un système RAG.
    """
    
    def __init__(self, embed_model):
        self.embed_model = embed_model
        self.documents = []
        self.embeddings = None
    
    def add_documents(self, documents):
        """
        Ajoute des documents à la base et calcule leurs embeddings.
        """
        self.documents = documents
        print(f"📚 Indexation de {len(documents)} documents...")
        self.embeddings = self.embed_model.encode(documents)
        print(f" Documents indexés ! Shape des embeddings : {self.embeddings.shape}")
    
    def retrieve(self, query, top_k=3):
        """
        Recherche les documents les plus pertinents pour une requête.
        """
        # Encoder la requête
        query_embedding = self.embed_model.encode([query])[0]
        
        # Calculer les similarités cosinus
        similarities = np.dot(self.embeddings, query_embedding) / (
            np.linalg.norm(self.embeddings, axis=1) * np.linalg.norm(query_embedding)
        )
        
        # Top-k indices
        top_indices = np.argsort(similarities)[-top_k:][::-1]
        
        results = []
        for idx in top_indices:
            results.append({
                'document': self.documents[idx],
                'score': similarities[idx]
            })
        
        return results
    
    def generate_prompt(self, query, top_k=3):
        """
        Génère un prompt augmenté avec les documents récupérés.
        """
        # Récupérer les documents pertinents
        retrieved_docs = self.retrieve(query, top_k)
        
        # Construire le contexte
        context = "\n\n".join([f"Document {i+1}: {doc['document']}" 
                               for i, doc in enumerate(retrieved_docs)])
        
        # Construire le prompt RAG
        prompt = f"""Utilise les documents suivants pour répondre à la question.
Si l'information n'est pas dans les documents, dis-le.

=== DOCUMENTS ==="
{context}

=== QUESTION ===
{query}

=== RÉPONSE ==="""
        
        return prompt, retrieved_docs

In [ ]:
# Base de connaissances exemple (sur une entreprise fictive)
knowledge_base = [
    "TechCorp a été fondée en 2010 par Alice Martin et Bob Dupont à Paris.",
    "Le siège social de TechCorp est situé au 42 rue de l'Innovation, 75001 Paris.",
    "TechCorp développe des solutions d'intelligence artificielle pour les entreprises.",
    "En 2023, TechCorp a réalisé un chiffre d'affaires de 50 millions d'euros.",
    "TechCorp emploie 200 personnes réparties dans 3 pays : France, Allemagne et Espagne.",
    "Le produit phare de TechCorp est 'SmartAssist', un assistant IA pour le service client.",
    "TechCorp a levé 30 millions d'euros en série B en 2022.",
    "Les principaux clients de TechCorp sont dans les secteurs bancaire et assurance.",
    "Alice Martin est la CEO de TechCorp, Bob Dupont est le CTO.",
    "TechCorp a remporté le prix de l'innovation IA en 2023."
]

# Créer le système RAG
rag = SimpleRAG(embed_model)
rag.add_documents(knowledge_base)

In [ ]:
# Test du RAG
questions = [
    "Qui a fondé TechCorp ?",
    "Quel est le chiffre d'affaires de TechCorp ?",
    "Où se trouve le siège de TechCorp ?",
    "Quel est le produit principal de TechCorp ?"
]

for question in questions:
    print("=" * 70)
    print(f" Question : {question}")
    print("-" * 70)
    
    # Récupérer les documents pertinents
    results = rag.retrieve(question, top_k=2)
    
    print(" Documents récupérés :")
    for i, res in enumerate(results):
        print(f"   {i+1}. (score: {res['score']:.3f}) {res['document']}")
    
    # Générer le prompt
    prompt, _ = rag.generate_prompt(question, top_k=2)
    print(f"\n Prompt généré pour le LLM :")
    print(prompt[:500] + "...")
    print()

## 5.2 Visualisation des embeddings

In [ ]:
from sklearn.manifold import TSNE

# Ajouter quelques requêtes pour visualiser
queries = [
    "fondateurs de l'entreprise",
    "localisation du siège",
    "résultats financiers"
]

# Encoder les requêtes
query_embeddings = embed_model.encode(queries)

# Combiner documents et requêtes
all_embeddings = np.vstack([rag.embeddings, query_embeddings])
all_labels = knowledge_base + queries
all_types = ['document'] * len(knowledge_base) + ['query'] * len(queries)

# Réduction de dimension avec t-SNE
tsne = TSNE(n_components=2, random_state=42, perplexity=5)
embeddings_2d = tsne.fit_transform(all_embeddings)

# Visualisation
plt.figure(figsize=(12, 8))

# Documents
doc_mask = np.array(all_types) == 'document'
plt.scatter(embeddings_2d[doc_mask, 0], embeddings_2d[doc_mask, 1], 
            c='blue', s=100, alpha=0.6, label='Documents')

# Requêtes
query_mask = np.array(all_types) == 'query'
plt.scatter(embeddings_2d[query_mask, 0], embeddings_2d[query_mask, 1], 
            c='red', s=200, marker='*', label='Requêtes')

# Labels
for i, (x, y) in enumerate(embeddings_2d):
    label = all_labels[i][:30] + "..." if len(all_labels[i]) > 30 else all_labels[i]
    plt.annotate(label, (x, y), fontsize=8, alpha=0.7)

plt.legend()
plt.title('Visualisation des embeddings (t-SNE)')
plt.xlabel('Dimension 1')
plt.ylabel('Dimension 2')
plt.tight_layout()
plt.show()

print("💡 Les requêtes (étoiles rouges) sont proches des documents sémantiquement similaires !")

---

# Partie 6 : Exploration des Capacités Émergentes

## 6.1 Test de différentes capacités

In [ ]:
def test_capability(name, prompt, expected_behavior):
    """
    Affiche un test de capacité pour discussion.
    """
    print("=" * 70)
    print(f" TEST : {name}")
    print("=" * 70)
    print(f"\n Prompt :\n{prompt}")
    print(f"\n Comportement attendu : {expected_behavior}")
    print("-" * 70)

# Test 1 : Arithmétique
test_capability(
    "Arithmétique multi-chiffres",
    "Calcule : 847 × 396 = ?",
    "Les petits modèles échouent, les grands (>10B params) réussissent"
)

# Test 2 : Chain-of-Thought
test_capability(
    "Chain-of-Thought Reasoning",
    """Jean a 3 fois plus de billes que Marie. 
Marie a 2 fois plus de billes que Pierre. 
Pierre a 4 billes. 
Combien de billes ont-ils en tout ?

Réfléchissons étape par étape.""",
    "Capacité émergente vers 100B paramètres"
)

# Test 3 : Theory of Mind
test_capability(
    "Theory of Mind (False Belief)",
    """Sally met une bille dans son panier et quitte la pièce.
Anne déplace la bille dans sa boîte pendant que Sally est partie.
Sally revient.
Où Sally va-t-elle chercher sa bille ?""",
    "GPT-4 réussit (~95%), GPT-3 échoue souvent (~40%)"
)

# Test 4 : Word Unscrambling
test_capability(
    "Word Unscrambling",
    "Réordonne les lettres pour former un mot : 'tnartopmi'",
    "'important' - Capacité émergente vers 100B params"
)

## 6.2 Comparaison de modèles de différentes tailles

In [ ]:
# Données simulées pour illustration des capacités émergentes
model_sizes = ['1B', '10B', '50B', '100B', '500B']
model_sizes_num = [1, 10, 50, 100, 500]

# Capacités et leurs "seuils d'émergence" simulés
capabilities = {
    'Arithmétique simple': [60, 75, 85, 90, 95],
    'Arithmétique complexe': [5, 15, 40, 85, 95],
    'Chain-of-Thought': [5, 10, 25, 80, 95],
    'Theory of Mind': [20, 35, 50, 90, 95],
    'Word Unscrambling': [5, 8, 15, 75, 90],
}

plt.figure(figsize=(12, 6))

for cap_name, scores in capabilities.items():
    plt.plot(model_sizes_num, scores, 'o-', linewidth=2, markersize=8, label=cap_name)

plt.xscale('log')
plt.xlabel('Nombre de paramètres (milliards)', fontsize=12)
plt.ylabel('Accuracy (%)', fontsize=12)
plt.title('Capacités Émergentes en fonction de la taille du modèle', fontsize=14)
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.xticks(model_sizes_num, model_sizes)
plt.ylim(0, 100)

# Ajouter une zone d'émergence
plt.axvspan(50, 150, alpha=0.1, color='red', label='Zone d\'émergence')
plt.text(75, 10, 'Zone d\'émergence\ntypique', fontsize=10, ha='center', color='red')

plt.tight_layout()
plt.show()

print("\n Observation : Les capacités émergentes montrent une transition abrupte")
print("   (courbe en S) plutôt qu'une amélioration graduelle.")

---

# Résumé et Points Clés

## Ce que vous avez appris dans ce TP :

1. **Tokenization BPE** : Algorithme de fusion itérative pour créer un vocabulaire optimal

2. **Attention Causale** : Mécanisme permettant à chaque token de ne voir que le passé
   - Formule : $\text{Attention}(Q,K,V) = \text{softmax}(\frac{QK^T}{\sqrt{d_k}} + M)V$

3. **Prompting** :
   - Zero-shot : Sans exemple
   - Few-shot : Avec quelques exemples
   - Chain-of-Thought : Raisonnement étape par étape

4. **RAG** : Retrieval-Augmented Generation pour réduire les hallucinations

5. **Capacités Émergentes** : Apparition abrupte de nouvelles capacités à grande échelle

---

## Pour aller plus loin

- Expérimentez avec l'API OpenAI ou Anthropic
- Essayez différents modèles HuggingFace
- Implémentez un système RAG complet avec FAISS
- Explorez les techniques avancées de prompting (Self-Consistency, Tree-of-Thought)

In [1]:
print("="*70)
print(" Vous avez terminé le TP Tutoriel !")
print("="*70)
print("\nPassez maintenant au TP Exercices pour mettre en pratique vos connaissances.")

 Vous avez terminé le TP Tutoriel !

Passez maintenant au TP Exercices pour mettre en pratique vos connaissances.
